# AMPS I · Week 05 — Lab
## Riesz, Lax–Milgram in matrices, assembly, PETSc KSP and MMS

| | |
|---|---|
| **Lecture sections** | Part A (Gram matrix, Riesz, coercivity) · Part B (assembly, KSP) · Part C (MMS) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | build the Gram matrix and a Riesz representer · compute coercivity constants as eigenvalues · assemble P1 matrices by hand and compare with DOLFINx · assemble and solve in parallel with PETSc · compare direct and iterative solvers · verify a solver with manufactured solutions |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx`
(and here also `scipy`, `petsc4py`) are used. Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The Gram matrix and the Riesz representer  *(lecture §A1–A3)*
On $V_h$ (P1, $8\times8$) the $H^1$ inner product is $\mathbf v^TG\mathbf u$ with $G=M+K$. For $L(v)=\int fv$, $f=xy$, the Riesz representer
solves $G\mathbf w=\mathbf F$, and $\|L\|_{V_h'}=\sqrt{\mathbf F^T\mathbf w}$. Small dense matrices, one process: `fem.assemble_matrix(...).to_dense()`.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

msh = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8)
V = fem.functionspace(msh, ("Lagrange", 1))
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
K = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_dense()
M = fem.assemble_matrix(fem.form(u * v * ufl.dx)).to_dense()
G = K + M
print("G symmetric:", np.allclose(G, G.T), "  smallest eigenvalue of G:", f"{np.linalg.eigvalsh(G).min():.3e}")

x = ufl.SpatialCoordinate(msh)
f = x[0] * x[1]
F = fem.assemble_vector(fem.form(f * v * ufl.dx)).array          # la.Vector → NumPy array
w = np.linalg.solve(G, F)                                         # Riesz representer
normL = np.sqrt(F @ w)
print(f"||L||_(V_h') = sqrt(F.w) = {normL:.6f}    ||f||_L2 = {np.sqrt(fem.assemble_scalar(fem.form(f * f * ufl.dx))):.6f}")

# the definition ||L|| = sup L(v)/||v||: other v give smaller ratios, v = w attains it
X = V.tabulate_dof_coordinates()
rng = np.random.default_rng(0)
for name, z in (("v = 1", np.ones(F.size)), ("v = x + y", X[:, 0] + X[:, 1]),
                ("v = random", rng.standard_normal(F.size)), ("v = w", w)):
    print(f"   {name:11s}: L(v)/||v|| = {F @ z / np.sqrt(z @ G @ z):.6f}")

# the Riesz representer IS the Helmholtz solution (-Δw + w = f, ∂w/∂n = 0)
problem = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + u * v * ufl.dx, f * v * ufl.dx,
                        petsc_options_prefix="riesz_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
uh = problem.solve()
print("max |w - u_h| =", np.abs(w - uh.x.array).max())

## 2 · Coercivity constants as eigenvalues  *(lecture §A5–A8)*
$\gamma_h=\min_{\mathbf v}\dfrac{\mathbf v^TK_b\mathbf v}{\mathbf v^TG\mathbf v}=\lambda_{\min}(K_b,G)$ for a symmetric form $b$ with matrix $K_b$.
Removing the boundary DOFs (rows and columns) gives the space with $v=0$ on $\partial\Omega$.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import scipy.linalg
import ufl
from dolfinx import fem, mesh

for N in (4, 8, 16):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    K = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_dense()
    M = fem.assemble_matrix(fem.form(u * v * ufl.dx)).to_dense()
    G = K + M
    msh.topology.create_connectivity(1, 2)
    bdofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    free = np.setdiff1d(np.arange(G.shape[0]), bdofs)
    lam = lambda A, B: scipy.linalg.eigh(A, B, eigvals_only=True)[0]
    print(f"N={N:2d}  Helmholtz {lam(K + M, G):.4f}   Poisson/Neumann {lam(K, G):+.1e}   "
          f"Poisson/Dirichlet {lam(K[np.ix_(free, free)], G[np.ix_(free, free)]):.4f}   |K @ ones| = {np.abs(K @ np.ones(len(K))).max():.1e}")
print("limit of the Dirichlet value, 2π²/(1+2π²) =", round(2 * np.pi**2 / (1 + 2 * np.pi**2), 4))

> **✏️ Your turn (5 min).** For $b_\kappa(u,v)=\int\nabla u\cdot\nabla v+\kappa uv$ (natural BC) compute $\gamma_h$ for $\kappa=0.1,\,1,\,10$ on the $8\times8$ mesh.
> Compare with your answer to Exercise A.1 ($\gamma=\min(1,\kappa)$). Which $v$ attains the minimum for $\kappa<1$?

In [ ]:
%%fenicsx -np 1
# TODO: gamma_h for kappa = 0.1, 1, 10

## 3 · Assembly by hand  *(lecture §B1)*
P1 on triangles: $\nabla\phi_i$ is constant on a cell, $K^e=|K|\,\nabla\Phi\,\nabla\Phi^T$, $M^e=\frac{|K|}{12}(\mathbf 1\mathbf 1^T+I)$.
The cell → DOF map is `V.dofmap.list` (Week 3). Compare with `fem.assemble_matrix`.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 6, 6, diagonal=mesh.DiagonalType.crossed)
V = fem.functionspace(msh, ("Lagrange", 1))
coords = V.tabulate_dof_coordinates()[:, :2]
n = coords.shape[0]
grad_hat = np.array([[-1.0, -1.0], [1.0, 0.0], [0.0, 1.0]])      # ∇̂φ̂_i of the reference P1 basis (rows)
Kh, Mh = np.zeros((n, n)), np.zeros((n, n))
for dofs in V.dofmap.list:                                       # loop over cells
    X = coords[dofs]                                             # 3 × 2 vertex coordinates
    J = np.column_stack((X[1] - X[0], X[2] - X[0]))              # Jacobian of F(ξ) = X0 + J ξ
    area = abs(np.linalg.det(J)) / 2
    grads = grad_hat @ np.linalg.inv(J)                          # rows: ∇φ_i = J^{-T} ∇̂φ̂_i
    Kh[np.ix_(dofs, dofs)] += area * grads @ grads.T
    Mh[np.ix_(dofs, dofs)] += area / 12 * (np.ones((3, 3)) + np.eye(3))

u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
K = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_dense()
M = fem.assemble_matrix(fem.form(u * v * ufl.dx)).to_dense()
print(f"{n} DOFs, {len(V.dofmap.list)} cells")
print("max |K_hand - K| =", np.abs(Kh - K).max(), "   max |M_hand - M| =", np.abs(Mh - M).max())

> **✏️ Your turn (5 min).** Add the load vector for $f=1$ to the loop ($F^e_i=\int_K\phi_i=|K|/3$) and compare with
> `fem.assemble_vector(fem.form(v * ufl.dx)).array`. What is $\sum_iF_i$, and why?

In [ ]:
%%fenicsx -np 1
# TODO: hand-assembled load vector for f = 1

## 4 · Parallel assembly and PETSc KSP  *(lecture §B2–B4)*
Helmholtz with the manufactured solution $\tilde u=\cos(\pi x)\cos(\pi y)$, P1, **two processes**.
Look at the right-hand side before and after `ghostUpdate`, then compare solvers.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector

def norms(msh, e):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * ufl.dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

comm = MPI.COMM_WORLD
for N in (32, 64, 128):
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])
    f = -ufl.div(ufl.grad(u_ex)) + u_ex
    a = fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + u * v * ufl.dx)
    L = fem.form(f * v * ufl.dx)

    A = assemble_matrix(a); A.assemble()
    b = assemble_vector(L)
    before = b.norm()                        # collective: every rank must call it
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    after = b.norm()
    if N == 32 and comm.rank == 0:
        print(f"||b|| before ghostUpdate {before:.6f}, after {after:.6f}\n")

    for name, ksp_type, pc_type in (("LU", "preonly", "lu"), ("CG+Jacobi", "cg", "jacobi"), ("CG+GAMG", "cg", "gamg")):
        ksp = PETSc.KSP().create(comm)
        ksp.setOperators(A); ksp.setType(ksp_type); ksp.getPC().setType(pc_type)
        ksp.setTolerances(rtol=1e-10)
        uh = fem.Function(V)
        ksp.solve(b, uh.x.petsc_vec)
        uh.x.scatter_forward()
        L2 = norms(msh, uh - u_ex)[0]
        if comm.rank == 0:
            print(f"N={N:3d} {name:9s}: iterations {ksp.getIterationNumber():4d}  reason {ksp.getConvergedReason():2d}  L2 error {L2:.4e}")
        ksp.destroy()
    A.destroy(); b.destroy()

> **✏️ Your turn (3 min).** Run the cell above with `-np 1`. Which numbers change, and which do not? Why is $\|b\|$ the same before and after
> `ghostUpdate` on one process?

> *your answer*

## 5 · Manufactured solutions and rates  *(lecture §C1–C2)*
Same problem, P1 and P2, $N=8,\dots,64$, LU. Results go to `/tmp/fnx/mms.npz` and are plotted in an ordinary cell.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def norms(msh, e):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * ufl.dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def table(Ns, E, label=""):
    E = np.asarray(E); r = np.log2(E[:-1] / E[1:])
    for i, N in enumerate(Ns):
        print(f"{label} N={N:3d}  L2 {E[i, 0]:.3e}  H1 {E[i, 1]:.3e}" + (f"   rates {r[i-1, 0]:.2f} {r[i-1, 1]:.2f}" if i else ""))

Ns = [8, 16, 32, 64]
out = {}
for k in (1, 2):
    E = []
    for N in Ns:
        msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
        V = fem.functionspace(msh, ("Lagrange", k))
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        x = ufl.SpatialCoordinate(msh)
        u_ex = ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])
        f = -ufl.div(ufl.grad(u_ex)) + u_ex
        problem = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + u * v * ufl.dx, f * v * ufl.dx,
                                petsc_options_prefix=f"mms{k}_{N}_",
                                petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
        uh = problem.solve()
        assert problem.solver.getConvergedReason() > 0
        E.append(norms(msh, uh - u_ex))
    out[f"P{k}"] = np.array(E)
    if MPI.COMM_WORLD.rank == 0:
        table(Ns, E, f"P{k}")
if MPI.COMM_WORLD.rank == 0:
    np.savez("/tmp/fnx/mms.npz", N=Ns, **out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/mms.npz")
h = 1 / d["N"]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, j, title in zip(axs, (0, 1), ("$L^2$ error", "$H^1$ seminorm error")):
    for k in (1, 2):
        ax.loglog(h, d[f"P{k}"][:, j], "o-", label=f"P{k}")
    ax.set_xticks(h, [f"1/{n}" for n in d["N"]]); ax.minorticks_off()
    ax.set_xlabel("$h$"); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

> **✏️ Your turn (5 min).** Replace $\tilde u$ by $\sin(\pi x)\cos(\pi y)$ (keep $f=-\Delta\tilde u+\tilde u$), P1 only. What happens to the errors and
> rates? Which boundary condition does $u_h$ satisfy, and which one does $\tilde u$ satisfy?

In [ ]:
%%fenicsx -np 2
# TODO: MMS with u~ = sin(pi x) cos(pi y), P1

## 6 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11 code for a %%fenicsx -np 2 cell. Verify a P2 solver for -Δu + u = f on the unit square with
du/dn = 0 by the method of manufactured solutions: u = cos(pi x) cos(pi y), f computed with UFL, solve with
CG + hypre BoomerAMG (rtol 1e-10), report the L2 and H1-seminorm errors and rates for N = 4, 8, 16, 32 on rank 0,
check that the solver converged, and print the CG iteration counts.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] Rates approach 3 ($L^2$) and 2 ($H^1$) and are steady over the last two refinements
- [ ] The exact solution is a UFL expression; $f$ is computed from it by UFL
- [ ] $\tilde u$ satisfies the boundary condition of the weak form ($\partial_n\tilde u=0$ here)
- [ ] `getConvergedReason() > 0` is checked, and LU gives the same errors
- [ ] The numbers are identical with `-np 1` and `-np 2`, and printed once